# Notebook 03_2 — Predictive Performance (Text-Only Embeddings)

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**  
Applied to: Amazon Women's Shoes (Size 8)

---

This notebook evaluates how well different model specifications predict:
- **Q_t** — Sales rank (demand proxy)
- **P_bb_t** — Buybox price

Model specifications compared (following paper Table 2):
- OLS / Boosting with Tabular features only
- OLS / Boosting with Tabular + PCA (5 components)
- OLS / Boosting with Tabular + Cluster Similarities
- Deep Time Independent (RoBERTa text embeddings)
- Deep Time Dependent (RoBERTa text + lag1 embeddings)

R² evaluated on both level (Q_t, P_bb_t) and first-difference (ΔQ_t, ΔP_bb_t) models.

## ① Mount Drive

In [ ]:
# ── Bucket-scoped output config (added by refactor) ─────────────────────
from pathlib import Path

def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError('Cannot find project root')

BRAND_BUCKET = 'women_top1_dream_pairs'   # ← change per bucket run
PROJECT_ROOT = _find_project_root()
DATA_ROOT    = PROJECT_ROOT / 'data' / BRAND_BUCKET
ANALYSIS_DIR = DATA_ROOT / 'analysis_datasets'
KEY_DIR      = DATA_ROOT / 'keys'
DATA_DIR     = DATA_ROOT / 'amzn_shoes_monthly_diffs_ffill_fixed_splits'
PRED_DIR     = DATA_ROOT / 'predictions'
OUTPUT_DIR   = DATA_ROOT / 'output' / 'predictive_performance_txt'
IMG_DIR      = PROJECT_ROOT / 'data' / 'images'   # shared (ASIN-unique)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'BRAND_BUCKET : {BRAND_BUCKET}')
print(f'OUTPUT_DIR   : {OUTPUT_DIR}')

## ② Set Working Directory

## ③ Imports

In [4]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)
print('✅ Imports done')

✅ Imports done


## ④ Load Dataset

Loads the text-only embeddings dataset from notebook 01_2.
`dropna()` removes first-period rows with no lag features.

In [5]:
txt_only = True
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"

df_full_train = pd.read_csv(ANALYSIS_DIR / f"{dataframe_name}_train.zip")
df_full_val   = pd.read_csv(ANALYSIS_DIR / f"{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val   = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val   = df_full_val.dropna()

dummy_subcat_names = [category for category in df_full_val["subcat_aggregated"].unique()]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Train shape: {df_full_train.shape}")
print(f"Val shape:   {df_full_val.shape}")
print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Train shape: (30876, 333)
Val shape:   (30888, 333)
Dummy subcat names: ['Other', 'Fashion Sneakers', 'Ballet & Dance', 'Walking', 'Flats', 'Mules & Clogs', 'Ankle & Bootie', 'Pumps', 'Road Running', 'Loafers & Slip-Ons', 'Heeled Sandals']
All time steps: ['2025-04-28', '2025-05-26', '2025-06-23', '2025-07-21', '2025-08-18', '2025-09-15', '2025-10-13', '2025-11-10', '2025-12-08', '2026-01-05', '2026-02-02', '2026-03-02']


Index(['ASIN', 'date', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=333)

## ⑤ Define Controls and Feature Specifications

Controls follow paper Section 3:
- Continuous: RATING, REVIEW_COUNT, offer counts
- Time dummies (excluding first period as baseline)
- Subcat dummies (excluding Residual as baseline)
- PCA components (5) and cluster similarities (5) as additional features

In [6]:
n_lags = 1

outcome   = "Q_t"
treatment = "P_bb_t"

outcome_diff   = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + ["Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = ["RATING", "REVIEW_COUNT"]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0", "similarity_cluster_1", "similarity_cluster_2",
    "similarity_cluster_3", "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls
additional_controls_deep = [var for var in additional_controls if var not in dummy_subcat_names]
controls_emb = [var for var in df_full_train.columns if "emb" in var]

print(f"Continuous controls:    {len(cont_controls)}")
print(f"Dummy controls:         {len(dummy_controls)}")
print(f"Total controls:         {len(additional_controls)}")
print(f"Embedding columns:      {len(controls_emb)}")

Continuous controls:    5
Dummy controls:         23
Total controls:         28
Embedding columns:      256


## ⑥ Initialize Results DataFrames

In [7]:
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df      = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)
print('✅ Results DataFrames initialized')

✅ Results DataFrames initialized


## ⑦ Deep Model R² — Level

Evaluates how well the deep learning predictions (from Part 5) explain
sales rank and price in both time-independent and lag1 configurations.

In [8]:
df_dict = {"Train": df_full_train, "Test": df_full_val}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome (time independent):  {r2_ml_l}")
    print(f"  R2 Treatment (time independent): {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"  R2 Outcome (lag1):               {r2_ml_l_lag1}")
    print(f"  R2 Treatment (lag1):             {r2_ml_m_lag1}")
    print()

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

results_df_deep

Evaluation for Train set
  R2 Outcome (time independent):  0.7982
  R2 Treatment (time independent): 0.7508
  R2 Outcome (lag1):               0.8903
  R2 Treatment (lag1):             0.7848

Evaluation for Test set
  R2 Outcome (time independent):  0.6639
  R2 Treatment (time independent): 0.6064
  R2 Outcome (lag1):               0.8892
  R2 Treatment (lag1):             0.6564



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.7982,0.6639,0.7508,0.6064
Deep Time Dependent,0.8903,0.8892,0.7848,0.6564


## ⑧ Deep Model R² — Diff

Same evaluation on first-difference outcomes (ΔQ_t, ΔP_bb_t).

In [9]:
results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome diff (time independent):  {r2_ml_l_diff}")
    print(f"  R2 Treatment diff (time independent): {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"  R2 Outcome diff (lag1):               {r2_ml_l_diff_lag1}")
    print(f"  R2 Treatment diff (lag1):             {r2_ml_m_diff_lag1}")
    print()

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

Evaluation for Train set
  R2 Outcome diff (time independent):  -8.116e-05
  R2 Treatment diff (time independent): -0.00588633
  R2 Outcome diff (lag1):               0.03946662
  R2 Treatment diff (lag1):             -0.0053144

Evaluation for Test set
  R2 Outcome diff (time independent):  -5.63e-05
  R2 Treatment diff (time independent): -0.00607467
  R2 Outcome diff (lag1):               0.03834605
  R2 Treatment diff (lag1):             -0.00640707



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,-0.000081,-0.000056,-0.005886,-0.006075
Deep Time Dependent,0.039467,0.038346,-0.005314,-0.006407


## ⑨ Build Feature Matrices

Constructs train and test feature matrices for tabular, PCA, and similarity
specifications used in OLS and Boosting models.

In [10]:
# Train set
y_train       = df_full_train[outcome].squeeze()
d_train       = df_full_train[treatment].squeeze()
y_train_diff  = df_full_train[outcome_diff].squeeze()
d_train_diff  = df_full_train[treatment_diff].squeeze()

x_train     = sm.add_constant(df_full_train[additional_controls])
x_train_pca = sm.add_constant(df_full_train[additional_controls + controls_pca])
x_train_sim = sm.add_constant(df_full_train[additional_controls + controls_similarities])

# Test set
y_test      = df_full_val[outcome].squeeze()
d_test      = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test     = sm.add_constant(df_full_val[additional_controls])
x_test_pca = sm.add_constant(df_full_val[additional_controls + controls_pca])
x_test_sim = sm.add_constant(df_full_val[additional_controls + controls_similarities])

# Rename columns for LightGBM (no special characters)
for df in [x_train, x_test, x_train_pca, x_test_pca, x_train_sim, x_test_sim]:
    df.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x), inplace=True)

print(f"x_train shape:     {x_train.shape}")
print(f"x_train_pca shape: {x_train_pca.shape}")
print(f"x_train_sim shape: {x_train_sim.shape}")

x_train shape:     (30876, 29)
x_train_pca shape: (30876, 34)
x_train_sim shape: (30876, 34)


## ⑩ Build Dict Structures

In [11]:
train_dict = {
    "y": y_train, "y_diff": y_train_diff,
    "d": d_train, "d_diff": d_train_diff,
    "x": x_train, "x_pca": x_train_pca, "x_sim": x_train_sim,
}

test_dict = {
    "y": y_test, "y_diff": y_test_diff,
    "d": d_test, "d_diff": d_test_diff,
    "x": x_test, "x_pca": x_test_pca, "x_sim": x_test_sim,
}
print('✅ Train and test dicts ready')

✅ Train and test dicts ready


## ⑪ Tabular Models — Level

Compares OLS and LightGBM Boosting across three feature specifications:
- **Tabular** — continuous + time/subcat dummies only
- **Tabular + PCA** — adds 5 PCA components from embeddings
- **Tabular + Similarities** — adds 5 cluster similarity scores

In [12]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y"])
    print(f"  R2 Outcome train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d"])
    print(f"  R2 Treatment train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_tab = pd.concat([results_df_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
]
results_df_tab.columns = column_names
results_df_tab.index   = row_names
results_df_tab


 Feature specification: x
  OLS
  R2 Outcome train/test: 0.1982 / 0.1688
  R2 Treatment train/test: 0.2404 / 0.1926
  Boosting
  R2 Outcome train/test: 0.5928 / 0.5222
  R2 Treatment train/test: 0.5621 / 0.4045

 Feature specification: x_pca
  OLS
  R2 Outcome train/test: 0.6312 / 0.5383
  R2 Treatment train/test: 0.6223 / 0.4988
  Boosting
  R2 Outcome train/test: 0.8906 / 0.7369
  R2 Treatment train/test: 0.8854 / 0.6857

 Feature specification: x_sim
  OLS
  R2 Outcome train/test: 0.6244 / 0.5380
  R2 Treatment train/test: 0.6282 / 0.5009
  Boosting
  R2 Outcome train/test: 0.8645 / 0.7153
  R2 Treatment train/test: 0.8511 / 0.6265


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.198218,0.168780,0.240419,0.192626
Boosting (Tabular),0.592756,0.522182,0.562144,0.404502
OLS (Tabular + PCA),0.631240,0.538274,0.622317,0.498846
Boosting (Tabular + PCA),0.890606,0.736908,0.885441,0.685739
OLS (Tabular + Similarities),0.624429,0.537963,0.628231,0.500931
Boosting (Tabular + Similarities),0.864541,0.715290,0.851078,0.626451


## ⑫ Summary — Level Models

In [13]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.198218,0.168780,0.240419,0.192626
Boosting (Tabular),0.592756,0.522182,0.562144,0.404502
OLS (Tabular + PCA),0.631240,0.538274,0.622317,0.498846
Boosting (Tabular + PCA),0.890606,0.736908,0.885441,0.685739
OLS (Tabular + Similarities),0.624429,0.537963,0.628231,0.500931
Boosting (Tabular + Similarities),0.864541,0.715290,0.851078,0.626451
Deep Time Independent,0.798200,0.663900,0.750800,0.606400
Deep Time Dependent,0.890300,0.889200,0.784800,0.656400


## ⑬ Tabular Models — Diff

Same comparison on first-difference outcomes (ΔQ_t, ΔP_bb_t).
First differences remove product fixed effects and time trends.

In [14]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y_diff"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome diff train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d_diff"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d_diff"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment diff train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y_diff"])
    print(f"  R2 Outcome diff train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d_diff"])
    print(f"  R2 Treatment diff train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_diff_tab = pd.concat([results_df_diff_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
]
results_df_diff_tab.columns = column_names
results_df_diff_tab.index   = row_names
results_df_diff_tab


 Feature specification: x
  OLS
  R2 Outcome diff train/test: 0.1634 / 0.1750
  R2 Treatment diff train/test: 0.0185 / 0.0149
  Boosting
  R2 Outcome diff train/test: 0.3649 / 0.2991
  R2 Treatment diff train/test: 0.1234 / 0.0517

 Feature specification: x_pca
  OLS
  R2 Outcome diff train/test: 0.1641 / 0.1754
  R2 Treatment diff train/test: 0.0187 / 0.0154
  Boosting
  R2 Outcome diff train/test: 0.4423 / 0.3291
  R2 Treatment diff train/test: 0.1810 / 0.0572

 Feature specification: x_sim
  OLS
  R2 Outcome diff train/test: 0.1641 / 0.1755
  R2 Treatment diff train/test: 0.0190 / 0.0157
  Boosting
  R2 Outcome diff train/test: 0.4218 / 0.3169
  R2 Treatment diff train/test: 0.1558 / 0.0529


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.163428,0.174993,0.018508,0.014902
Boosting (Tabular),0.364878,0.299111,0.123402,0.051698
OLS (Tabular + PCA),0.164122,0.175401,0.018717,0.015362
Boosting (Tabular + PCA),0.442291,0.329128,0.180971,0.057177
OLS (Tabular + Similarities),0.164052,0.175461,0.018956,0.015674
Boosting (Tabular + Similarities),0.421806,0.316868,0.155792,0.052876


## ⑭ Summary — Diff Models

In [15]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.163428,0.174993,0.018508,0.014902
Boosting (Tabular),0.364878,0.299111,0.123402,0.051698
OLS (Tabular + PCA),0.164122,0.175401,0.018717,0.015362
Boosting (Tabular + PCA),0.442291,0.329128,0.180971,0.057177
OLS (Tabular + Similarities),0.164052,0.175461,0.018956,0.015674
Boosting (Tabular + Similarities),0.421806,0.316868,0.155792,0.052876
Deep Time Independent,-0.000081,-0.000056,-0.005886,-0.006075
Deep Time Dependent,0.039467,0.038346,-0.005314,-0.006407


## ⑮ Final Summary (% format)

R² values multiplied by 100 for readability — matches paper Table 2 format.

In [16]:
print("=== Level Models — Test R² (%) ===")
print(results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100)
print()
print("=== Diff Models — Test R² (%) ===")
print(results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100)

=== Level Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          16.88      19.26
Boosting (Tabular)                     52.22      40.45
OLS (Tabular + PCA)                    53.83      49.88
Boosting (Tabular + PCA)               73.69      68.57
OLS (Tabular + Similarities)           53.80      50.09
Boosting (Tabular + Similarities)      71.53      62.65
Deep Time Independent                  66.39      60.64
Deep Time Dependent                    88.92      65.64

=== Diff Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          17.50       1.49
Boosting (Tabular)                     29.91       5.17
OLS (Tabular + PCA)                    17.54       1.54
Boosting (Tabular + PCA)               32.91       5.72
OLS (Tabular + Similarities)           17.55       1.57
Boosting (Tabular + Similarities)      31.69       5.29
Deep Time Independent             